# E3 · Cabeza MLP con pérdida triplet sobre E2 congelado

E3 pregunta si una proyección aprendida con identidades públicas mejora la reidentificación de mascotas sin ajustar el detector ni el encoder de E2.

| Elemento | Diseño fijado |
|---|---|
| Objetivo | Medir la transferencia de una cabeza MLP entrenada sobre identidades públicas. |
| Pregunta | ¿La proyección 512→256→512 mejora Rank-k y mAP frente a E2 en Public v2 y PetTrace v5? |
| Hipótesis | Batch-hard triplet incrementará la separación entre identidades de la misma especie. |
| Intervención | Entrenar solo la MLP; detector YOLO y encoder CLIP permanecen congelados. |
| Controles | E1 y E2 publicados, verificados sobre las mismas consultas y galerías. |
| Muestra | Corpus público refinado v2 completo, equivalente a la preparación v4.100. |
| Selección | mAP macro por especie en la validación pública; el test no selecciona checkpoints. |
| Evaluación | Public v2 y PetTrace v5, con galerías separadas, closed-set y misma especie. |
| Repeticiones | Semillas 45–49, fijadas antes de ejecutar. |
| Handoff E4/E5 | Gana la mayor mAP macro de validación; un empate exacto favorece la menor semilla. |

Las cinco semillas comparten datos, features e hiperparámetros; se selecciona la ganadora **antes** de evaluar los dos tests.


### Entorno reproducible

Ejecutar en una sesión nueva de **Google Colab con GPU T4**. El paquete fija el código; Hugging Face aporta los datasets inmutables y MLflow conserva resultados y artefactos. No se usa Google Drive ni se repite la limpieza de datos.

Los secretos `HF_TOKEN`, `MLFLOW_TRACKING_USERNAME` y `MLFLOW_TRACKING_PASSWORD` deben existir en la UI de Colab y tener acceso habilitado para este notebook.


In [ ]:
%pip install -q --no-cache-dir "pettrace-ml[training,notebooks,tracking]==0.2.37"


In [ ]:
import json, os, shutil, sys

# Configure quiet, non-interactive downloads before importing Hub clients.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import torch
from IPython.display import Markdown, display
from google.colab import userdata

import pettrace
from pettrace.tracking import PRODUCTION_URI, source_identity

if sys.platform != "linux" or not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("E3 Colab profile requires Linux with Python 3.11–3.13")
if pettrace.__version__ != "0.2.37":
    raise RuntimeError("Install PetTrace 0.2.37 before continuing")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime; a T4 is sufficient")

print("Access [1/2] Loading Colab Secrets…")
os.environ["MLFLOW_TRACKING_URI"] = PRODUCTION_URI
for name in ("HF_TOKEN", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    value = userdata.get(name)
    if not isinstance(value, str) or not value.strip():
        raise TypeError(f"Colab Secret {name} must be a non-empty text value")
    os.environ[name] = value
print("Access [2/2] Credentials available.")

EXPERIMENT, DATASET = "E3", "v5"
DATASET_COMMIT = "1777734dbffda0df0bf43305233809ee73157d94"
TRAINING_DATASET_VERSION, PREPARED_VERSION = "v2", "v4.100"
SOURCE_SAMPLE_PCT, SEEDS = 100, (45, 46, 47, 48, 49)
BASE_MODEL = "models:/pettrace-embedder/2"
ENCODER_SHA256 = "4773e649274ca8774c4a63d45f0438fe598ee7ba67e65def97537aa0eca9a576"
DETECTOR_SHA256 = "da8f9b9ebea5157cdcefa7bea9fd782249ebdf119f917c07d64e5b41942b9c9d"
LOG_TO_MLFLOW = True
REGISTER_E3 = True

RUN_LABEL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
SWEEP_REPORT = Path("/content/reports/e3_v5") / RUN_LABEL
SWEEP_REPORT.mkdir(parents=True)
REPORT = SWEEP_REPORT
FEATURE_CACHE = Path("/content/cache/pettrace/e3")
os.environ["PETTRACE_EXECUTION_ID"] = RUN_LABEL
SOURCE_IDENTITY = source_identity()
SOURCE_SHA256 = SOURCE_IDENTITY["source_sha256"]
mlflow.set_tracking_uri(PRODUCTION_URI)
mlflow.set_registry_uri(PRODUCTION_URI)
print("Package:", pettrace.__version__, "| GPU:", torch.cuda.get_device_name(0),
      "| Seeds:", SEEDS, "| Report:", REPORT)


## Fase 1. Preparación y verificación de datos

El notebook **no limpia ni vuelve a ejecutar YOLO**. Descarga las releases públicas refinadas `v2`, fijadas por commit, verifica manifests, hashes, splits y cajas YOLO, y materializa la copia local necesaria para esta sesión.

El entrenamiento usa el corpus público completo. Public v2 y PetTrace v5 se reservan para evaluación; PetTrace nunca participa en entrenamiento o selección.


In [ ]:
from pettrace.data import training
from pettrace.data.volume import training_volume
from pettrace.experiments.e3 import load_benchmarks
from pettrace.tracking import production_context

print("Dataset [1/3] Loading the immutable public v2 corpus…")
prepared = training(
    version=TRAINING_DATASET_VERSION,
    cache_dir="/content/datasets/pettrace-refined",
    token=os.environ["HF_TOKEN"],
)
manifest_rows = prepared.rows
training_source = prepared.provenance
coverage_metrics, coverage_params = training_volume(prepared, SOURCE_SAMPLE_PCT, REPORT)
(REPORT / "transport-equivalence.json").write_text(json.dumps(training_source, indent=2))

print("Dataset [2/3] Verifying MLflow and the two immutable benchmarks…")
production_context()
benchmarks = load_benchmarks(
    prepared,
    REPORT,
    private_version=DATASET,
    private_commit=DATASET_COMMIT,
)
private_ds, public_ds = benchmarks.private, benchmarks.public
PUBLIC_REPORT = benchmarks.public_report

rows = [
    (source.species, source.repo_id, source.commit[:12],
     sum(row["source"] == source.source for row in manifest_rows),
     len({row["animal"] for row in manifest_rows if row["source"] == source.source}))
    for source in prepared.sources
]
table = ["| Especie | Release refinada | Commit | Fotos retenidas | Identidades |",
         "|---|---|---|---:|---:|"]
table += [f"| {species} | `{repo}` | `{commit}` | {photos:,} | {identities:,} |"
          for species, repo, commit, photos, identities in rows]
table.append(f"| **Total** | `v2` / `{PREPARED_VERSION}` | `{prepared.fingerprint[:12]}` | "
             f"**{len(manifest_rows):,}** | **{prepared.summary['retained_identities']:,}** |")
display(Markdown("\n".join(table)))
print("Dataset [3/3] Ready:", len(public_ds.photos), "public test photos ·",
      len(private_ds.photos), "PetTrace photos. Exact-image and identity leakage checks passed.")


## Fase 2. Representación y entrenamiento

E2 permanece congelado: YOLO selecciona el animal y CLIP ViT-B/32 genera un embedding L2 de 512 dimensiones. E3 aprende únicamente una MLP `512 → 256 → 512`, con ReLU y normalización L2.

La pérdida batch-hard triplet sigue a [Hermans, Beyer y Leibe (2017)](https://arxiv.org/abs/1703.07737): dentro de cada lote se usa el positivo más lejano y el negativo más cercano. Los lotes alternan especies y contienen 8 identidades × 4 fotos. AdamW usa `lr=1e-3`, `weight_decay=1e-4`, margen `0.2`, máximo 30 épocas y paciencia 7.

Los embeddings congelados se calculan una sola vez. Después se entrenan las semillas 45–49 y se fija la ganadora exclusivamente por mAP macro de validación. En este punto ningún test ha sido evaluado.


In [ ]:
from pettrace.data.verification import sha256
from pettrace.evaluation.core import load_model
from pettrace.experiments.e3 import select_validation_candidate
from pettrace.models.cuda import cuda_session
from pettrace.models.golden import compare, golden_images
from pettrace.training.features import feature_cache
from pettrace.training.triplet import Config, export_encoder, fit
from pettrace.ui import markdown_table

base_model = load_model(BASE_MODEL)
if not base_model.golden_ok or base_model.pipeline.detector is None:
    raise RuntimeError("The frozen E2 control failed golden verification")
if base_model.sha256 != {"encoder.onnx": ENCODER_SHA256, "detector.onnx": DETECTOR_SHA256}:
    raise RuntimeError("The frozen E2 artifacts changed")

gpu_cache = Path("/content/cache/pettrace/onnx-cuda")
gpu_detector, detector_gpu = cuda_session(
    base_model.pipeline.detector, Path(base_model.dir) / "detector.onnx", gpu_cache)
gpu_pipeline, encoder_gpu = cuda_session(
    base_model.pipeline, Path(base_model.dir) / "encoder.onnx", gpu_cache)
gpu_pipeline.detector = gpu_detector
golden_ok, minimum_cosine, maximum_error = compare(
    np.load(Path(base_model.dir) / "golden.npy"), gpu_pipeline.embed(golden_images()))
if not golden_ok:
    raise RuntimeError("CUDA and frozen E2 outputs are not equivalent")
base_model.pipeline = gpu_pipeline

training_rows = [row for row in manifest_rows if row["split"] in {"train", "val"}]
features, cache_digest = feature_cache(
    base_model.pipeline, training_rows, FEATURE_CACHE, base_model.dir, batch=128)
print("Feature cache ready:", features.shape, "| SHA-256:", cache_digest[:12])

candidates = []
for seed in SEEDS:
    seed_report = REPORT / f"seed-{seed}"
    seed_report.mkdir()
    for filename in ("dataset-coverage.json", "sources.json", "transport-equivalence.json"):
        shutil.copy2(REPORT / filename, seed_report / filename)
    config = Config(seed=seed, epochs=30, steps_per_epoch=100,
                    identities_per_batch=8, images_per_identity=4, hidden=256,
                    margin=0.2, learning_rate=1e-3, weight_decay=1e-4, patience=7)
    run_config = {
        "experiment": EXPERIMENT,
        "seed": seed,
        "seed_sweep": list(SEEDS),
        "selection_metric": "validation.macro_species_mAP",
        "training": asdict(config),
        "prepared_dataset_fingerprint": prepared.fingerprint,
        "training_dataset_source": training_source,
        "base_model": base_model.uri,
        "base_model_sha256": base_model.sha256,
        "embedding_gpu": {"detector": detector_gpu, "encoder": encoder_gpu,
                          "golden_min_cosine": minimum_cosine,
                          "golden_max_abs_diff": maximum_error},
        "feature_cache_sha256": cache_digest,
        "source": SOURCE_IDENTITY,
        "log_to_mlflow": LOG_TO_MLFLOW,
        "register_selected_model": REGISTER_E3,
    }
    (seed_report / "training-config.json").write_text(json.dumps(run_config, indent=2))
    print(f"\nTraining seed {seed} · validation only; tests remain unread")
    trained_head, train_summary = fit(
        features, training_rows, seed_report, config=config, device="cuda")
    (seed_report / "training-summary.json").write_text(json.dumps(train_summary, indent=2))
    model_dir = seed_report / "model"
    onnx_hashes = export_encoder(
        base_model.dir, trained_head, model_dir,
        model_version=f"clip-vit-b32-crop-mlp-e3-{PREPARED_VERSION}-seed{seed}@v3")
    candidates.append({
        "seed": seed,
        "best_epoch": train_summary["best_epoch"],
        "best_val_mAP": train_summary["best_val_mAP"],
        "config": config,
        "training_summary": train_summary,
        "report": seed_report,
        "model_dir": model_dir,
        "model_sha256": onnx_hashes,
    })

selected = select_validation_candidate(candidates)
selection = {
    "selected_before_test_evaluation": True,
    "metric": "validation.macro_species_mAP",
    "tie_break": "lower_seed",
    "seeds": list(SEEDS),
    "candidates": [{key: candidate[key] for key in ("seed", "best_epoch", "best_val_mAP")}
                   for candidate in candidates],
    "selected_seed": selected["seed"],
}
(REPORT / "selection.json").write_text(json.dumps(selection, indent=2))
for candidate in candidates:
    shutil.copy2(REPORT / "selection.json", candidate["report"] / "selection.json")
display(Markdown("### Selección por validación\n\n" + markdown_table(
    ["Seed", "Mejor época", "mAP macro de validación", "Seleccionado"],
    [[candidate["seed"], candidate["best_epoch"], f"{candidate['best_val_mAP']:.4f}",
      "Sí" if candidate is selected else "No"] for candidate in candidates],
)))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for candidate in candidates:
    history = candidate["training_summary"]["history"]
    epochs = [row["epoch"] for row in history]
    emphasis = candidate["seed"] == selected["seed"]
    axes[0].plot(epochs, [row["train.loss"] for row in history],
                 alpha=1 if emphasis else .35, linewidth=2.2 if emphasis else 1,
                 label=f"seed {candidate['seed']}")
    axes[1].plot(epochs, [row["val.macro_species_mAP"] for row in history],
                 alpha=1 if emphasis else .35, linewidth=2.2 if emphasis else 1,
                 label=f"seed {candidate['seed']}")
axes[0].set(title="Optimización multisemilla", xlabel="Época", ylabel="Triplet loss")
axes[1].set(title="Selección sin observar test", xlabel="Época", ylabel="mAP macro de validación")
for axis in axes:
    axis.grid(alpha=.2)
axes[1].legend(frameon=False, ncol=2)
fig.tight_layout()
fig.savefig(REPORT / "training-curves.png", dpi=180, bbox_inches="tight")
plt.show()


## Fase 3. Evaluación y análisis

Con la selección ya congelada, las cinco semillas se evalúan y publican en dos dominios independientes. Cada foto es una consulta; la galería contiene las otras fotos de la misma especie. Rank-k agrega el máximo score por identidad y mAP conserva el ranking por fotografías. EER se estima sobre los mismos pares evaluados y por ello es **descriptiva**, no una calibración independiente.

El resultado principal del barrido es la media y desviación estándar de las cinco semillas. Las figuras E1→E2→E3 y el análisis por identidad usan la semilla elegida por validación para el handoff a E4/E5. Los tests describen desempeño; nunca alteran esa elección.


In [ ]:
from pettrace.evaluation.core import evaluate, load_model
from pettrace.evaluation.figures import save
from pettrace.evaluation.research import save_benchmark_comparison
from pettrace.experiments.e3 import frozen_controls, publish_results
from pettrace.tracking import save_provenance

for candidate in candidates:
    seed = candidate["seed"]
    seed_report = candidate["report"]
    seed_public_report = seed_report / "public_test"
    seed_public_report.mkdir(exist_ok=True)
    shutil.copy2(PUBLIC_REPORT / "dataset-selection.json",
                 seed_public_report / "dataset-selection.json")
    e3_model = load_model(str(candidate["model_dir"]))
    if not e3_model.golden_ok:
        raise RuntimeError(f"The exported E3 seed {seed} model failed golden verification")

    print(f"\nSeed {seed} · evaluating frozen Public v2 and PetTrace v5 benchmarks")
    public_ev = evaluate(public_ds, e3_model.pipeline, zero_shot=True)
    private_ev = evaluate(private_ds, e3_model.pipeline, zero_shot=True)
    save(public_ev, e3_model, str(seed_public_report))
    save(private_ev, e3_model, str(seed_report))
    save_provenance(seed_public_report, e3_model)
    publication = publish_results(
        public_ev, private_ev, e3_model, seed_report, seed_public_report, SOURCE_SHA256,
        experiment=EXPERIMENT, source_sample_pct=SOURCE_SAMPLE_PCT,
        config=candidate["config"], training_summary=candidate["training_summary"],
        prepared_fingerprint=prepared.fingerprint, training_source=training_source,
        coverage_metrics=coverage_metrics, coverage_params=coverage_params,
        private_dataset_version=DATASET,
    )
    candidate.update({"model": e3_model, "public_evaluation": public_ev,
                      "private_evaluation": private_ev,
                      "public_report": seed_public_report, "publication": publication})
    print("Published · public:", publication["public_run_id"],
          "· PetTrace:", publication["run_id"])

selected = next(candidate for candidate in candidates if candidate["seed"] == selection["selected_seed"])
REPORT, PUBLIC_REPORT = selected["report"], selected["public_report"]
e3_model = selected["model"]
public_ev, private_ev = selected["public_evaluation"], selected["private_evaluation"]
config, train_summary = selected["config"], selected["training_summary"]

public_controls, public_control_summaries = frozen_controls(public_ev, "public", PUBLIC_REPORT)
private_controls, private_control_summaries = frozen_controls(private_ev, "private", REPORT)
comparison_markdown, domain_figure = save_benchmark_comparison(
    public_ev, private_ev, REPORT, EXPERIMENT)
display(Markdown(comparison_markdown))
display(domain_figure)
plt.close(domain_figure)


In [ ]:
domains = {
    "Public v2": {**public_controls, "E3": public_ev},
    "PetTrace v5": {**private_controls, "E3": private_ev},
}
metrics = (("rank1", "Rank-1"), ("rank2", "Rank-2"), ("mAP", "mAP"))
models = ("E1", "E2", "E3")
colors = {"Rank-1": "#0F766E", "Rank-2": "#60A5FA", "mAP": "#B45309"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
table = ["| Dominio | Modelo | Rank-1 | Rank-2 | mAP |", "|---|---|---:|---:|---:|"]
for axis, (domain, evaluations) in zip(axes, domains.items()):
    x = np.arange(len(models)); width = .24
    for offset, (key, label) in enumerate(metrics):
        values = [evaluations[model].result["test"]["global"][key] * 100 for model in models]
        bars = axis.bar(x + (offset - 1) * width, values, width, label=label, color=colors[label])
        axis.bar_label(bars, fmt="%.1f", fontsize=8, padding=2)
    axis.set(title=domain, xticks=x, xticklabels=models, xlabel="Experimento", ylim=(0, 105))
    axis.grid(axis="y", alpha=.2)
    for model in models:
        values = evaluations[model].result["test"]["global"]
        table.append(f"| {domain} | {model} | {values['rank1']:.2%} | "
                     f"{values['rank2']:.2%} | {values['mAP']:.2%} |")
axes[0].set_ylabel("Resultado (%)")
axes[1].legend(frameon=False, ncol=3, loc="lower center")
fig.suptitle("Evolución del pipeline con consultas y galerías congeladas")
fig.tight_layout()
fig.savefig(REPORT / "e1-e2-e3-comparison.png", dpi=180, bbox_inches="tight")
display(Markdown("\n".join(table)))
plt.show()


In [ ]:
def identity_metrics(evaluation):
    grouped = {}
    for row in evaluation.result["test"]["per_query"]:
        grouped.setdefault(row["animal"], []).append(row)
    return {animal: {"rank1": float(np.mean([row["rank"] == 1 for row in rows])),
                     "mAP": float(np.mean([row["ap"] for row in rows]))}
            for animal, rows in grouped.items()}

identity_analysis = {}
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for axis, (domain, controls, current) in zip(
        axes,
        (("Public v2", public_controls, public_ev),
         ("PetTrace v5", private_controls, private_ev))):
    before, after = identity_metrics(controls["E2"]), identity_metrics(current)
    if set(before) != set(after):
        raise ValueError("E2 and E3 must evaluate the same identities")
    deltas = {
        metric: np.array([(after[animal][metric] - before[animal][metric]) * 100
                          for animal in sorted(after)])
        for metric in ("rank1", "mAP")
    }
    bins = np.linspace(min(deltas["rank1"].min(), deltas["mAP"].min()),
                       max(deltas["rank1"].max(), deltas["mAP"].max()), 12)
    axis.hist(deltas["rank1"], bins=bins, alpha=.6, label="Rank-1", color="#0F766E")
    axis.hist(deltas["mAP"], bins=bins, alpha=.6, label="mAP", color="#B45309")
    axis.axvline(0, color="#374151", linestyle="--")
    axis.set(title=domain, xlabel="Cambio E3 − E2 (puntos porcentuales)")
    identity_analysis[domain] = {
        metric: {"mean_pp": float(values.mean()), "median_pp": float(np.median(values)),
                 "improved": int((values > 0).sum()), "identities": len(values)}
        for metric, values in deltas.items()
    }
axes[0].set_ylabel("Identidades")
axes[1].legend(frameon=False)
fig.suptitle("Heterogeneidad del efecto por identidad")
fig.tight_layout()
fig.savefig(REPORT / "identity-deltas.png", dpi=180, bbox_inches="tight")
plt.show()

analysis = {
    "protocol": "closed_set_same_species_photo_leave_one_out",
    "training_dataset_fingerprint": prepared.fingerprint,
    "coverage": coverage_metrics,
    "training": train_summary,
    "identity_deltas_e3_minus_e2": identity_analysis,
}
(REPORT / "analysis.json").write_text(json.dumps(analysis, indent=2))
display(Markdown("`identity-deltas.png` resume la distribución; no identifica causas ni pondera "
                 "por número de fotos. Los valores por identidad permanecen en los resultados de evaluación."))


In [ ]:
from statistics import mean, stdev

new_results = []
for candidate in candidates:
    public = candidate["public_evaluation"].result["test"]["global"]
    private = candidate["private_evaluation"].result["test"]["global"]
    publication = candidate["publication"]
    new_results.append({
        "seed": candidate["seed"], "validation_mAP": candidate["best_val_mAP"],
        "public": {key: public[key] for key in ("rank1", "rank2", "mAP")},
        "private": {key: private[key] for key in ("rank1", "rank2", "mAP")},
        "public_run": publication["public_run_id"], "private_run": publication["run_id"],
    })

lines = ["| Seed | Validación mAP | Público Rank-1 | mAP | PetTrace Rank-1 | mAP | MLflow |",
         "|---:|---:|---:|---:|---:|---:|---|"]
for row in new_results:
    selected_mark = " **★**" if row["seed"] == selected["seed"] else ""
    links = (f"[público](https://ml.pettrace.app/#/experiments/3/runs/{row['public_run']}) · "
             f"[PetTrace](https://ml.pettrace.app/#/experiments/3/runs/{row['private_run']})")
    lines.append(f"| {row['seed']}{selected_mark} | {row['validation_mAP']:.4f} | "
                 f"{row['public']['rank1']:.2%} | {row['public']['mAP']:.2%} | "
                 f"{row['private']['rank1']:.2%} | {row['private']['mAP']:.2%} | {links} |")
lines += ["", "| Dominio | Rank-1 medio ± DE | Rank-2 medio ± DE | mAP medio ± DE |",
          "|---|---:|---:|---:|"]
for domain, label in (("public", "Public v2"), ("private", "PetTrace v5")):
    summary = {key: [row[domain][key] for row in new_results]
               for key in ("rank1", "rank2", "mAP")}
    lines.append(f"| {label} | {mean(summary['rank1']):.2%} ± {stdev(summary['rank1'])*100:.2f} pp | "
                 f"{mean(summary['rank2']):.2%} ± {stdev(summary['rank2'])*100:.2f} pp | "
                 f"{mean(summary['mAP']):.2%} ± {stdev(summary['mAP'])*100:.2f} pp |")
display(Markdown("\n".join(lines)))
(SWEEP_REPORT / "seed-study.json").write_text(json.dumps({
    "selection": selection, "results": new_results,
    "interpretation": "training variability; not population uncertainty",
}, indent=2))
print("Selected for E4/E5 by validation only: seed", selected["seed"])


### Discusión y límites

La evidencia multisemilla reporta media y desviación estándar como resultado principal. La semilla marcada no se presenta como una estimación superior de la población: fue seleccionada por la validación fijada para proporcionar un único control a E4/E5. Los resultados de test no intervienen en esa decisión. Las diferencias entre público y PetTrace son descriptivas porque cambian identidades, tamaño de galería y condiciones de captura.

El protocolo es closed-set: presupone que la identidad correcta está en la galería y que la especie es conocida. No estima búsqueda abierta, rendimiento a escala de ciudad ni generalización entre sesiones independientes. Las etiquetas públicas y el filtro YOLO reducen ruido, pero no verifican manualmente cada identidad. EER se calcula sobre los pares reportados y no constituye una calibración externa.


### Publicación obligatoria y registro separado

La evaluación anterior publica obligatoriamente las cinco semillas, cada una con runs separados para Public v2 y PetTrace v5. Si la red falla, los modelos y reportes ya entrenados permanecen en el runtime y la publicación puede reintentarse sin volver a entrenar. La celda siguiente registra **solo** la semilla seleccionada por validación, sin promover aliases de servicio.


In [ ]:
from pettrace.experiments.e3 import register_candidate

if REGISTER_E3:
    publication = selected["publication"]
    registration = register_candidate(
        selected["report"], publication["run_id"], publication["public_run_id"])
    handoff = {
        "experiment": "E3", "selected_seed": selected["seed"],
        "selection_metric": selection["metric"],
        "validation_mAP": selected["best_val_mAP"],
        "public_run_id": publication["public_run_id"],
        "private_run_id": publication["run_id"],
        "model_name": registration["name"], "model_version": registration["version"],
        "alias_promoted": False,
    }
    handoff_path = selected["report"] / "e3-handoff.json"
    handoff_path.write_text(json.dumps(handoff, indent=2))
    mlflow.MlflowClient().log_artifact(
        publication["run_id"], str(handoff_path), artifact_path="report")
    print("E4/E5 control:", handoff, "· no alias promoted")


### Referencias

- Radford et al. (2021), [Learning Transferable Visual Models From Natural Language Supervision](https://arxiv.org/abs/2103.00020): encoder CLIP congelado.
- Hermans, Beyer y Leibe (2017), [In Defense of the Triplet Loss for Person Re-Identification](https://arxiv.org/abs/1703.07737): pérdida batch-hard; su evidencia en personas no demuestra eficacia en mascotas.
- Ultralytics, [YOLOv8](https://docs.ultralytics.com/models/yolov8/): localización previa del animal.
- [Dogs of the World](https://www.kaggle.com/datasets/lextoumbourou/dogs-world) y [LCW](https://www.kaggle.com/datasets/dseidli/lcwlabeled-cats-in-the-wild): fuentes de identidades públicas; este trabajo usa sus releases refinadas y versionadas en Hugging Face.
